# LTLf Discrete SAC training

This self-contained Kaggle notebook installs the dependencies, reconstructs every DSAC source file, runs training, displays the learning metrics and packages the outputs. No repository clone, Kaggle Dataset or additional source upload is required.

Enable **Internet** and optionally select a **GPU accelerator** in the Kaggle notebook settings before running all cells.


## 1. Install system and Python dependencies


In [ ]:
!apt-get update -qq
!apt-get install -y -qq mona graphviz swig
%pip install -q "gymnasium[box2d]" "tianshou==0.5.1" ltlf2dfa graphviz pandas matplotlib

import gymnasium
import tianshou
import torch

print(f"PyTorch: {torch.__version__}")
print(f"Tianshou: {tianshou.__version__}")
print(f"Gymnasium: {gymnasium.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


## 2. Create the writable project directory


In [ ]:
from pathlib import Path
import os

WORK_DIR = Path("/kaggle/working/dsac")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")


## 3. Reconstruct the DSAC source files


### `abstract_mdps.py`


In [ ]:
%%writefile abstract_mdps.py
import re
from collections import defaultdict
import numpy as np

# Import the LTLf parser provided by ltlf2dfa.
from ltlf2dfa.parser.ltlf import LTLfParser
from graphviz import Source

class LTLfAutomaton:
    """
    Wrap ltlf2dfa and expose its DFA as a graph that can be traversed by the MDP.
    """
    def __init__(self, formula_str):
        self.formula_str = formula_str
        
        # Parse the formula and generate its DFA in DOT format.
        parser = LTLfParser()
        parsed_formula = parser(formula_str)
        dot_string = parsed_formula.to_dfa()
        self.dot_string = parsed_formula.to_dfa()
        
        # Initialize the automaton data structures.
        self.states = set()
        self.accepting_states = set()
        self.transitions = {}  # {source_state: [(Boolean_guard, destination_state), ...]}
        self.initial_state = None
        
        # Extract states and transitions from the DOT representation.
        self._parse_dot(dot_string)
        
        # Keep a stable state order for the MDP and one-hot encodings.
        self.states = sorted(list(self.states))
        self.num_phases = len(self.states)

    def _parse_dot(self, dot_string):
        """
        Parse the DOT output and extract states, accepting states, the initial
        state, and guarded transitions.
        """
        # Extract accepting states, e.g. node [shape = doublecircle]; 2 3;.
        match_acc = re.search(r'node\s*\[shape\s*=\s*doublecircle\]\s*;\s*(.*?);', dot_string)
        if match_acc:
            acc_str = match_acc.group(1).replace(',', ' ')
            self.accepting_states = set(int(s) for s in acc_str.split() if s.strip().isdigit())
            
        # Extract guarded transitions, e.g. 1 -> 2 [label="wp1 & ~wp2"].
        trans_matches = re.findall(r'(\d+)\s*->\s*(\d+)\s*\[label\s*=\s*"(.*?)"\]', dot_string)
        for src_str, dst_str, guard in trans_matches:
            src = int(src_str)
            dst = int(dst_str)
            self.states.add(src)
            self.states.add(dst)
            
            if src not in self.transitions:
                self.transitions[src] = []
            self.transitions[src].append((guard, dst))
            
        # Extract the initial state from the unlabeled edge leaving the invisible node.
        # Example: 0 [style=invis]; 0 -> 1;.
        init_match = re.search(r'(\d+)\s*->\s*(\d+)\s*;', dot_string)
        if init_match:
            self.initial_state = int(init_match.group(2))
        else:
            self.initial_state = min(self.states) if self.states else 0

    def get_initial_q(self):
        """Return the identifier of the DFA pre-trace state."""
        return self.initial_state

    def is_goal_reached(self, current_q):
        """Return whether the current DFA state is accepting."""
        return current_q in self.accepting_states

    def get_next_q(self, current_q, truth_assignment):
        """
        Evaluate outgoing transition guards and return the next DFA state.
        """
        if current_q not in self.transitions:
            return current_q
            
        for guard, next_q in self.transitions[current_q]:
            if self._eval_guard(guard, truth_assignment):
                return next_q
                
        return current_q

    def _eval_guard(self, guard, truth_assignment):
        """
        Convert a DOT guard such as "wp1 & ~wp2" to Python syntax and evaluate
        it against the current truth assignment.
        """
        guard = guard.strip()
        
        # Handle numeric and textual Boolean constants.
        if guard.lower() in ["1", "true"]: return True
        if guard.lower() in ["0", "false"]: return False
        
        # Convert the standard Boolean operators to Python syntax.
        expr = guard.replace('&', ' and ').replace('|', ' or ').replace('~', ' not ').replace('!', ' not ')
        
        try:
            # Disable built-ins while evaluating the Boolean expression.
            return eval(expr, {"__builtins__": {}}, truth_assignment)
        except Exception as e:
            print(f"[LTLfAutomaton error] Could not evaluate transition guard '{guard}': {e}")
            return False

    def render_graph(self, filename="ltlf_automaton", directory="img"):
        """Render the DFA and save it as a PNG image."""
        try:
            # ltlf2dfa emits a left-to-right graph.  With complex formulae the
            # transition guards become wide, leaving the resulting PNG only a
            # few pixels high.  A top-to-bottom layout gives labels enough room
            # and keeps the automaton readable independently of formula length.
            render_dot = re.sub(
                r"rankdir\s*=\s*LR\s*;",
                "rankdir = TB;",
                self.dot_string,
                count=1,
            )
            render_dot = re.sub(
                r"(digraph[^{]*\{)",
                (
                    r"\1\n"
                    r'graph [pad="0.35", nodesep="0.55", ranksep="0.75"];' "\n"
                    r'node [width="0.55", height="0.55"];' "\n"
                    r'edge [fontsize="10"];'
                ),
                render_dot,
                count=1,
            )
            src = Source(render_dot)
            src.render(filename=filename, directory=directory, format='png', cleanup=True)
            print(f"Automaton graph saved to: {directory}/{filename}.png")
        except Exception as e:
            print(f"[Graphviz error] Could not render the automaton graph: {e}")


class LTLfWaypointMDP:
    """
    Abstract MDP guided by an LTLf automaton.
    Each abstract state is (x, y, q), where q is the DFA state identifier.
    """
    def __init__(self, waypoints_dict, ltlf_automaton, width=12, height=12, gamma=0.99, goal_reward=10000):
        self.width = width
        self.height = height
        self.gamma = gamma
        self.actions = [0, 1, 2, 3, 4, 5, 6, 7] # Include diagonal movements.
        
        self.waypoints_dict = waypoints_dict
        self.automaton = ltlf_automaton
        self.num_phases = self.automaton.num_phases
        
        # Generate every combination of grid position and DFA state.
        self.states = [(x, y, q) for x in range(width) for y in range(height) for q in self.automaton.states]
        
        self.goal_reward = goal_reward
        self.v_star = defaultdict(float)
        
    def _get_truth_assignment(self, x, y):
        """
        Map the current grid coordinates to a Boolean proposition assignment.
        """
        truth_assignment = {}
        for prop_name, (wp_x, wp_y) in self.waypoints_dict.items():
            truth_assignment[prop_name] = (x == wp_x and y == wp_y)
        return truth_assignment

    def get_transitions(self, state, action):
        x, y, q = state
        reward = 0
        
        # Apply the abstract physical movement.
        next_y = y
        if action in [0, 4, 5]:    next_y = min(y + 1, self.height - 1)
        elif action in [1, 6, 7]:  next_y = max(y - 1, 0)
            
        next_x = x
        if action in [2, 4, 6]:    next_x = max(x - 1, 0)
        elif action in [3, 5, 7]:  next_x = min(x + 1, self.width - 1)
        
        # Evaluate propositions at the arrival coordinates.
        truth_assignment = self._get_truth_assignment(next_x, next_y)
        
        # Advance the automaton using the arrival-state valuation.
        next_q = self.automaton.get_next_q(q, truth_assignment)

        next_state = (next_x, next_y, next_q)
        return next_state, reward

    def print_policy(self):
        arrows = {
            0: "↑",
            1: "↓",
            2: "←",
            3: "→",
            4: "↖",
            5: "↗",
            6: "↙",
            7: "↘"
        }

        for q in self.automaton.states:
            print(f"\n===== POLICY - DFA STATE q={q} =====")

            for y in reversed(range(self.height)):
                row = []

                for x in range(self.width):
                    state = (x, y, q)

                    if self.automaton.is_goal_reached(q):
                        row.append(" G ")
                        continue

                    best_action = None
                    best_value = -float("inf")

                    for a in self.actions:
                        next_state, reward = self.get_transitions(state, a)
                        value = reward + self.gamma * self.v_star[next_state]

                        if value > best_value:
                            best_value = value
                            best_action = a

                    row.append(f" {arrows[best_action]} ")

                print("".join(row))
    
    def value_iteration(self, theta=0.001):
        print(f"Value Iteration...")
        
        for s in self.states:
            if self.automaton.is_goal_reached(s[2]):
                self.v_star[s] = self.goal_reward
        
        while True:
            delta = 0
            new_v = self.v_star.copy()
            for s in self.states:
                if not self.automaton.is_goal_reached(s[2]):
                    v_actions = [self.get_transitions(s, a)[1] + self.gamma * self.v_star[self.get_transitions(s, a)[0]] for a in self.actions]
                    best_v = max(v_actions)
                    delta = max(delta, abs(best_v - self.v_star[s]))
                    new_v[s] = best_v
            self.v_star = new_v
            if delta < theta: break

        #self.print_policy()
        


### `automaton_validator.py`


In [ ]:
%%writefile automaton_validator.py
"""DFA validation shared with the LunarLander trainers."""

import itertools
import re
from collections import deque


LTLF_OPERATORS = {"F", "G", "M", "R", "U", "W", "X", "false", "true"}


def _extract_formula_propositions(formula):
    tokens = set(re.findall(r"[A-Za-z_][A-Za-z0-9_]*", formula))
    return sorted(token for token in tokens if token not in LTLF_OPERATORS)


def _generate_truth_assignments(propositions):
    for values in itertools.product((False, True), repeat=len(propositions)):
        yield dict(zip(propositions, values))


def _matching_transitions(automaton, state, truth_assignment):
    return [
        (guard, destination)
        for guard, destination in automaton.transitions.get(state, [])
        if automaton._eval_guard(guard, truth_assignment)
    ]


class AutomatonValidationReport:
    def __init__(self, formula, propositions):
        self.formula = formula
        self.propositions = propositions
        self.errors = []
        self.warnings = []
        self.statistics = {}

    @property
    def is_valid(self):
        return not self.errors

    def add_error(self, message):
        self.errors.append(message)

    def add_warning(self, message):
        self.warnings.append(message)

    def format(self):
        status = "VALID" if self.is_valid else "INVALID"
        lines = [
            "=== AUTOMATON VALIDATION ===",
            f"Status: {status}",
            f"Formula propositions: {self.propositions}",
            f"Statistics: {self.statistics}",
        ]
        if self.errors:
            lines.append("Errors:")
            lines.extend(f"- {message}" for message in self.errors)
        if self.warnings:
            lines.append("Warnings:")
            lines.extend(f"- {message}" for message in self.warnings)
        return "\n".join(lines)

    def raise_if_invalid(self):
        if not self.is_valid:
            raise ValueError(self.format())


def validate_automaton(
    automaton,
    waypoints_dict,
    width=None,
    height=None,
    max_propositions=12,
    raise_on_error=True,
):
    """Validate DFA structure, guards, reachability, propositions and coordinates."""
    propositions = _extract_formula_propositions(automaton.formula_str)
    report = AutomatonValidationReport(automaton.formula_str, propositions)
    states = set(automaton.states)
    waypoint_propositions = set(waypoints_dict)

    missing_waypoints = sorted(set(propositions) - waypoint_propositions)
    unused_waypoints = sorted(waypoint_propositions - set(propositions))
    if missing_waypoints:
        report.add_error(f"Formula propositions without coordinates: {missing_waypoints}")
    if unused_waypoints:
        report.add_warning(f"Waypoint propositions not used by the formula: {unused_waypoints}")
    if len(propositions) > max_propositions:
        report.add_error(
            f"The formula has {len(propositions)} propositions; "
            f"exhaustive validation is limited to {max_propositions}"
        )

    for proposition, coordinates in waypoints_dict.items():
        if not isinstance(coordinates, (tuple, list)) or len(coordinates) != 2:
            report.add_error(f"Waypoint {proposition!r} must contain exactly two coordinates")
            continue
        x, y = coordinates
        if not isinstance(x, int) or not isinstance(y, int):
            report.add_error(f"Waypoint {proposition!r} coordinates must be integers")
        if width is not None and not 0 <= x < width:
            report.add_error(f"Waypoint {proposition!r} has x={x}, outside [0, {width - 1}]")
        if height is not None and not 0 <= y < height:
            report.add_error(f"Waypoint {proposition!r} has y={y}, outside [0, {height - 1}]")

    if not states:
        report.add_error("The DFA contains no states")
    if automaton.get_initial_q() not in states:
        report.add_error(
            f"Initial state {automaton.get_initial_q()!r} is not part of the DFA"
        )
    unknown_accepting = sorted(set(automaton.accepting_states) - states)
    if unknown_accepting:
        report.add_error(f"Unknown accepting states: {unknown_accepting}")
    if not automaton.accepting_states:
        report.add_error("The DFA has no accepting states")
    for source, transitions in automaton.transitions.items():
        if source not in states:
            report.add_error(f"Transition source {source!r} is not part of the DFA")
        for guard, destination in transitions:
            if destination not in states:
                report.add_error(
                    f"Transition {source!r} --[{guard}]--> {destination!r} "
                    "targets an unknown state"
                )

    truth_assignments = (
        list(_generate_truth_assignments(propositions))
        if len(propositions) <= max_propositions
        else []
    )
    reachable_states = (
        {automaton.get_initial_q()} if automaton.get_initial_q() in states else set()
    )
    frontier = deque(reachable_states)
    checked_pairs = 0
    ambiguous_pairs = 0
    incomplete_pairs = 0

    for state in states:
        for truth_assignment in truth_assignments:
            matches = _matching_transitions(automaton, state, truth_assignment)
            checked_pairs += 1
            if not matches:
                incomplete_pairs += 1
                report.add_error(
                    f"No transition from state {state!r} for valuation {truth_assignment}"
                )
            elif len(matches) > 1:
                ambiguous_pairs += 1
                report.add_error(
                    f"Ambiguous transitions from state {state!r} for valuation "
                    f"{truth_assignment}: {[guard for guard, _ in matches]}"
                )
            else:
                expected_destination = matches[0][1]
                actual_destination = automaton.get_next_q(state, truth_assignment)
                if actual_destination != expected_destination:
                    report.add_error(
                        f"get_next_q returned {actual_destination!r}, expected "
                        f"{expected_destination!r} from state {state!r}"
                    )

    while frontier and truth_assignments:
        state = frontier.popleft()
        for truth_assignment in truth_assignments:
            matches = _matching_transitions(automaton, state, truth_assignment)
            if len(matches) == 1 and matches[0][1] not in reachable_states:
                reachable_states.add(matches[0][1])
                frontier.append(matches[0][1])

    unreachable_states = sorted(states - reachable_states)
    unreachable_accepting = sorted(set(automaton.accepting_states) - reachable_states)
    if unreachable_states:
        report.add_warning(f"Unreachable DFA states: {unreachable_states}")
    if unreachable_accepting:
        report.add_error(f"No accepting path reaches states: {unreachable_accepting}")

    report.statistics = {
        "states": len(states),
        "accepting_states": len(automaton.accepting_states),
        "transitions": sum(len(transitions) for transitions in automaton.transitions.values()),
        "valuations": len(truth_assignments),
        "state_valuation_pairs": checked_pairs,
        "ambiguous_pairs": ambiguous_pairs,
        "incomplete_pairs": incomplete_pairs,
        "reachable_states": len(reachable_states),
    }
    if raise_on_error:
        report.raise_if_invalid()
    return report


### `utils.py`


In [ ]:
%%writefile utils.py
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def phi_mapping_grid(obs, grid_w=12, grid_h=12):
    x, y = obs[0], obs[1]
    abstract_x = int(np.clip((x + 1) / 2 * (grid_w - 1), 0, grid_w - 1))
    abstract_y = int(np.clip(y / 1.5 * (grid_h - 1), 0, grid_h - 1))
    return abstract_x, abstract_y

def phi_mapping_sequential(obs, q, grid_w=12, grid_h=12):
    abstract_x, abstract_y = phi_mapping_grid(obs, grid_w, grid_h)
    return abstract_x, abstract_y, q

def save_sequential_heatmaps(abstract_mdp, filename_prefix="v_star"):
    """
    Generates and saves a separate heatmap for V* for each phase defined in the MDP,
    without any waypoint or goal markers (clean heatmap).
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    # Store every heatmap directly under img/heatmaps.
    output_dir = os.path.join("img", "heatmaps")
    os.makedirs(output_dir, exist_ok=True)
    filename_prefix = os.path.basename(filename_prefix)
    
    width, height = abstract_mdp.width, abstract_mdp.height
    
    # Extract global min/max for consistent colormap scaling
    all_values = np.array(list(abstract_mdp.v_star.values()))
    computed_vmin = all_values.min() if len(all_values) > 0 else 0
    computed_vmax = all_values.max() if len(all_values) > 0 else 1

    for current_q in abstract_mdp.automaton.states:
        matrix = np.zeros((height, width))
        for (x, y, q), value in abstract_mdp.v_star.items():
            if q == current_q and 0 <= x < width and 0 <= y < height:
                matrix[y, x] = value
                
        plt.figure(figsize=(9, 8))
        im = plt.imshow(matrix, cmap='viridis', origin='lower', vmin=computed_vmin, vmax=computed_vmax)
        
        for y in range(height):
            for x in range(width):
                val = matrix[y, x]
                if val > 0.0: 
                    text_color = 'white' if val < (computed_vmax / 2) else 'black'
                    plt.text(x, y, f"{val:.1f}", ha='center', va='center', color=text_color, fontsize=7)
                    
        plt.colorbar(im, fraction=0.046, pad=0.04, label="Potential Value (V*)")
        
        is_goal_state = abstract_mdp.automaton.is_goal_reached(current_q)
        phase_label = "Goal Reached" if is_goal_state else "Seeking Targets"
        plt.title(f"Potential Map (V*) - DFA State q={current_q} ({phase_label})", fontsize=14, fontweight='bold')
        
        ax = plt.gca()
        ax.set_xticks(np.arange(-.5, width, 1), minor=True)
        ax.set_yticks(np.arange(-.5, height, 1), minor=True)
        ax.grid(which='minor', color='w', linestyle='-', linewidth=1, alpha=0.4)
        
        # Keep the heatmap free of waypoint and goal markers.
            
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{filename_prefix}_q{current_q}.png"), dpi=150, bbox_inches='tight')
        plt.close()
        print(f" -> Generated V* Heatmap for DFA State q={current_q}")

def plot_comparison_curves(baseline_rewards, shaping_rewards, epsilon_history=None, window_size=100, filename="img/baseline_vs_shaping.png", title="Learning Curve Comparison", baseline_label="Baseline", shaping_label="Shaping"):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax1 = plt.subplots(figsize=(12, 7))
    baseline_ma = pd.Series(baseline_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
    shaping_ma = pd.Series(shaping_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
    x_axis = np.arange(len(baseline_rewards))
    ax1.plot(x_axis, baseline_ma, color='black', linestyle='-', linewidth=2, label=baseline_label)
    ax1.plot(x_axis, shaping_ma, color='blue', linestyle='-', linewidth=2.5, label=shaping_label)
    ax1.set_title(title, fontsize=15, fontweight='bold')
    ax1.set_xlabel(f"Episode # (Moving Average Window = {window_size})", fontsize=12)
    ax1.set_ylabel("Episode Reward", fontsize=12)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    if epsilon_history:
        ax2 = ax1.twinx()
        ax2.plot(x_axis, epsilon_history, color='orange', linestyle='--', linewidth=1.8, label='Epsilon Decay')
        ax2.set_ylabel("Exploration Rate (ε)", color='orange', fontsize=12)
        ax2.tick_params(axis='y', labelcolor='orange')
        ax2.set_ylim(0, 1.05)
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right", fontsize=11)
    else:
        ax1.legend(loc="lower right", fontsize=11)

    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    print(f"\n>>> Comparison plot successfully saved to: {filename}")
    plt.close(fig)

def plot_mean_std_curves(reward_histories_single=None, reward_histories_multi=None, window_size=100, title="Mean Performance with Variance", filename="img/mean_std_plot.png"):
    """
    Plots the mean and standard deviation of reward histories for one or two sets of runs.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax = plt.subplots(figsize=(12, 7))

    def plot_single_curve(reward_histories, label, color):
        if not reward_histories:
            return
        
        # Ensure all histories have the same length by padding with NaNs if necessary
        max_len = max(len(h) for h in reward_histories)
        padded_histories = [np.pad(h, (0, max_len - len(h)), 'constant', constant_values=np.nan) for h in reward_histories]
        
        rewards_df = pd.DataFrame(padded_histories).T
        mean_rewards = rewards_df.mean(axis=1)
        std_rewards = rewards_df.std(axis=1)

        # Apply moving average
        mean_ma = mean_rewards.rolling(window=window_size, min_periods=1, center=True).mean()
        std_ma = std_rewards.rolling(window=window_size, min_periods=1, center=True).mean()

        x_axis = np.arange(len(mean_ma))
        ax.plot(x_axis, mean_ma, label=f"Mean {label}", color=color, linewidth=2.5)
        ax.fill_between(x_axis, mean_ma - std_ma, mean_ma + std_ma, color=color, alpha=0.2, label=f"Std Dev {label}")

    if reward_histories_single:
        plot_single_curve(reward_histories_single, "Single Epsilon", "black")

    if reward_histories_multi:
        plot_single_curve(reward_histories_multi, "Multi Epsilon", "blue")

    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.set_xlabel(f"Episode # (Moving Average Window = {window_size})", fontsize=12)
    ax.set_ylabel("Mean Episode Reward", fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc="lower right", fontsize=11)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    print(f"\n>>> Mean/Std plot successfully saved to: {filename}")
    plt.close(fig)

def plot_buffer_fractions(buffer_histories, window_size=100, filename="img/buffer_fractions.png", state_labels=None):
    """
    Plots the replay buffer composition for N phases dynamically.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    x_axis = np.arange(len(buffer_histories[0]))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(buffer_histories)))
    for idx, history in enumerate(buffer_histories):
        ma = pd.Series(history).rolling(window=window_size, min_periods=1, center=True).mean()
        state_label = state_labels[idx] if state_labels is not None else idx
        ax.plot(x_axis, ma, color=colors[idx], linewidth=2.5, label=f'DFA state q={state_label}')
    
    ax.set_title(f"Replay Buffer Composition (MA Window = {window_size})", fontsize=14, fontweight='bold')
    ax.set_ylabel("Fraction in Buffer", fontsize=12)
    ax.set_ylim(0, 1.05)
    
    ideal_balance = 1.0 / len(buffer_histories)
    ax.axhline(y=ideal_balance, color='gray', linestyle=':', alpha=0.7, label=f'Ideal Balance ({ideal_balance:.0%})')
    
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=len(buffer_histories)+1, fontsize=11)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    plt.close(fig)

def plot_shaping_reward_breakdown(true_rewards, total_rewards, eps_histories, window_size=100, filename="img/shaping_reward_breakdown.png"):
    """
    Plots the moving average of rewards (True vs Total) and overlays the N-phase Epsilon decay dynamically.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax1 = plt.subplots(figsize=(12, 7))
    
    # Moving Average Calculation
    if len(true_rewards) >= window_size:
        true_ma = pd.Series(true_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
        total_ma = pd.Series(total_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
    else:
        true_ma = true_rewards
        total_ma = total_rewards
        
    x_axis = np.arange(len(true_rewards))
        
    # Plot Rewards (Left Y-Axis)
    ax1.plot(x_axis, true_ma, color='green', linestyle='-', linewidth=2, label='Synthetic Goal Reward')
    ax1.plot(x_axis, total_ma, color='purple', linestyle='-', linewidth=2.5, label='Learning Reward (Goal + Shaping)')
    
    ax1.set_title(f"Shaping Agent Reward Analysis (MA Window = {window_size})", fontsize=15, fontweight='bold')
    ax1.set_xlabel("Episode #", fontsize=12)
    ax1.set_ylabel("Episode Reward", fontsize=12)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Plot Epsilon Decays (Right Y-Axis)
    ax2 = ax1.twinx()
    
    # Check if eps_histories is a list of lists/arrays (multi-epsilon case)
    is_multi_eps = any(isinstance(i, (list, np.ndarray)) for i in eps_histories)

    if is_multi_eps:
        num_phases = len(eps_histories)
        colors = plt.cm.plasma(np.linspace(0, 0.8, num_phases))
        for idx in range(num_phases):
            label = "Goal" if idx == num_phases - 1 else f"WP {idx + 1}"
            ax2.plot(x_axis, eps_histories[idx], color=colors[idx], linestyle='--', linewidth=2, alpha=0.8, label=f'ε Decay (q={idx}: {label})')
    else: # Single epsilon history
        ax2.plot(x_axis, eps_histories, color='orange', linestyle='--', linewidth=1.8, label='Epsilon Decay')

    # Align the zero of both y-axes for better visual comparison
    y1_min, y1_max = ax1.get_ylim()
    y2_min, y2_max = -0.05, 1.05 # Epsilon range is fixed
    
    # Align y-axes so that the zero points match.
    if y1_min < 0 < y1_max:
        # Calculate the proportional position of zero on the reward axis
        zero_ratio = -y1_min / (y1_max - y1_min)
        # Set the epsilon axis limits so its zero is at the same ratio
        new_y2_min = -zero_ratio * y2_max / (1 - zero_ratio)
        ax2.set_ylim(new_y2_min, y2_max)
    else:
        ax2.set_ylim(y2_min, y2_max)

    ax2.set_ylabel("Exploration Rate (ε)", color='black', fontsize=12)

    # Combine Legends from both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    
    # Dynamically calculate legend columns based on number of items to keep it compact
    legend_cols = max(2, (len(labels1) + len(labels2)) // 2)
    
    ax1.legend(
        lines1 + lines2, labels1 + labels2, 
        loc="upper center", bbox_to_anchor=(0.5, -0.15), 
        ncol=legend_cols, fontsize=11, framealpha=1.0
    )
    
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    plt.close(fig)


### `dsac_metrics.py`


In [ ]:
%%writefile dsac_metrics.py
"""Episode metrics collected by the LTLf environment wrapper."""

from collections import defaultdict

import numpy as np


# =============================================================================
# Mutable training state
# =============================================================================

class TrainingMetrics:
    """Own all mutable training metrics without relying on module globals."""

    def __init__(self, expected_episodes=None, log_interval=100):
        self.expected_episodes = expected_episodes
        self.log_interval = log_interval
        self.enabled = False
        self.clear()

    def clear(self):
        """Discard all recorded episodes and DFA transitions."""
        self.task_rewards = []
        self.total_rewards = []
        self.episode_lengths = []
        self.episode_end_reasons = []
        self.dfa_transition_counts = defaultdict(int)

    def record_transition(self, source, destination):
        """Count a DFA transition when metric collection is enabled."""
        if self.enabled:
            self.dfa_transition_counts[(source, destination)] += 1

    def record_episode(self, task_reward, total_reward, length, end_reason, use_shaping):
        """Append one completed episode and print periodic diagnostics."""
        if not self.enabled:
            return
        self.task_rewards.append(task_reward)
        self.total_rewards.append(total_reward)
        self.episode_lengths.append(length)
        self.episode_end_reasons.append(end_reason)
        if len(self.task_rewards) % self.log_interval == 0:
            self._print_progress(use_shaping)

    def _print_progress(self, use_shaping):
        """Print recent and cumulative learning diagnostics."""
        window = min(self.log_interval, len(self.task_rewards))
        recent_rewards = np.asarray(self.task_rewards[-window:])
        recent_reasons = self.episode_end_reasons[-window:]
        successes = int(np.count_nonzero(np.asarray(self.task_rewards) > 0))
        transitions = ", ".join(
            f"{source}->{destination}: {count}"
            for (source, destination), count in sorted(self.dfa_transition_counts.items())
        ) or "none"
        mode = "SHAPING" if use_shaping else "BASELINE"
        total = self.expected_episodes if self.expected_episodes is not None else "?"
        training_reward_line = (
            f"Average training reward      : {np.mean(self.total_rewards[-window:]):.6f}\n"
            if use_shaping
            else ""
        )
        print(
            f"\n[{mode} | DSAC] Episode {len(self.task_rewards)}/{total}\n"
            f"Average task reward          : {np.mean(recent_rewards):.6f}\n"
            f"Cumulative successes         : {successes}/{len(self.task_rewards)} "
            f"({successes / len(self.task_rewards):.2%})\n"
            f"Success rate (last {window}) : {np.mean(recent_rewards > 0):.2%}\n"
            f"Endings (last {window})      : "
            f"environment={recent_reasons.count('environment_terminated')}, "
            f"truncated={recent_reasons.count('truncated')}, "
            f"success={recent_reasons.count('success')}\n"
            f"Average episode length       : {np.mean(self.episode_lengths[-window:]):.1f}\n"
            f"{training_reward_line}"
            f"DFA transitions (cumulative) : {transitions}"
        )

    def validate_episode_count(self, expected_episodes):
        """Ensure that every metric contains exactly one value per episode."""
        lengths = {
            "task_rewards": len(self.task_rewards),
            "total_rewards": len(self.total_rewards),
            "episode_lengths": len(self.episode_lengths),
            "episode_end_reasons": len(self.episode_end_reasons),
        }
        if any(length != expected_episodes for length in lengths.values()):
            raise RuntimeError(
                f"Expected metrics for {expected_episodes} episodes, recorded {lengths}."
            )

    def as_dict(self):
        """Return defensive copies suitable for serialization."""
        return {
            "task_rewards": self.task_rewards.copy(),
            "total_rewards": self.total_rewards.copy(),
            "episode_lengths": self.episode_lengths.copy(),
            "episode_end_reasons": self.episode_end_reasons.copy(),
            "dfa_transition_counts": dict(self.dfa_transition_counts),
        }


### `dsac_environment.py`


In [ ]:
%%writefile dsac_environment.py
"""Gymnasium wrapper that augments observations with an LTLf DFA state."""

import gymnasium as gym
import numpy as np

from utils import phi_mapping_sequential


# =============================================================================
# LTLf observation and reward wrapper
# =============================================================================

class LTLfShapingWrapper(gym.Wrapper):
    """Apply the synthetic task reward and potential-based reward shaping."""

    def __init__(
        self,
        env,
        abstract_mdp,
        metrics,
        use_shaping=True,
        shaping_scale=1.0,
        goal_reward=10000.0,
    ):
        super().__init__(env)
        self.abstract_mdp = abstract_mdp
        self.metrics = metrics
        self.use_shaping = use_shaping
        self.shaping_scale = shaping_scale
        self.goal_reward = goal_reward
        self.automaton_states = list(abstract_mdp.automaton.states)
        self.state_to_index = {
            state: index for index, state in enumerate(self.automaton_states)
        }
        self.current_dfa_state = None
        self.previous_observation = None
        self.episode_task_reward = 0.0
        self.episode_total_reward = 0.0
        self.episode_length = 0

        # The policy receives the physical LunarLander state followed by a
        # one-hot encoding of the active DFA state.
        observation_size = env.observation_space.shape[0] + len(self.automaton_states)
        self.observation_space = gym.spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(observation_size,),
            dtype=np.float32,
        )

    @staticmethod
    def _abstract_position(observation):
        x, y, _ = phi_mapping_sequential(observation, 0)
        return x, y

    def _augment_observation(self, observation, dfa_state):
        one_hot = np.zeros(len(self.automaton_states), dtype=np.float32)
        one_hot[self.state_to_index[dfa_state]] = 1.0
        return np.concatenate((observation, one_hot)).astype(np.float32)

    def reset(self, **kwargs):
        """Reset the environment and evaluate the initial observation in the DFA."""
        observation, info = self.env.reset(**kwargs)
        x, y = self._abstract_position(observation)
        valuation = self.abstract_mdp._get_truth_assignment(x, y)

        # The DFA initial node represents the empty trace. Consuming s0 here
        # ensures that the first policy observation carries the correct state.
        pre_trace_state = self.abstract_mdp.automaton.get_initial_q()
        self.current_dfa_state = self.abstract_mdp.automaton.get_next_q(
            pre_trace_state,
            valuation,
        )
        self.previous_observation = observation
        self.episode_task_reward = 0.0
        self.episode_total_reward = 0.0
        self.episode_length = 0
        return self._augment_observation(observation, self.current_dfa_state), info

    def step(self, action):
        """Advance the environment, DFA and potential-based reward process."""
        # The native LunarLander reward is intentionally discarded: this
        # experiment learns exclusively from the temporal task and shaping.
        observation, _, terminated, truncated, info = self.env.step(action)
        environment_terminated = terminated
        self.episode_length += 1

        x, y = self._abstract_position(self.previous_observation)
        next_x, next_y = self._abstract_position(observation)
        valuation = self.abstract_mdp._get_truth_assignment(next_x, next_y)
        next_dfa_state = self.abstract_mdp.automaton.get_next_q(
            self.current_dfa_state,
            valuation,
        )

        task_reward = 0.0
        task_success = False

        # Entering an accepting DFA state completes the temporal task and ends
        # the Gymnasium episode with the configured synthetic goal reward.
        if next_dfa_state != self.current_dfa_state:
            self.metrics.record_transition(self.current_dfa_state, next_dfa_state)
            if self.abstract_mdp.automaton.is_goal_reached(next_dfa_state):
                task_reward = self.goal_reward
                task_success = True
                terminated = True

        abstract_state = (x, y, self.current_dfa_state)
        next_abstract_state = (next_x, next_y, next_dfa_state)
        shaping_reward = 0.0

        # Potential-based shaping is applied only when the complete abstract
        # state changes, using F(s,s') = K * (gamma * V*(s') - V*(s)).
        if self.use_shaping and abstract_state != next_abstract_state:
            current_potential = self.abstract_mdp.v_star.get(abstract_state, 0.0)
            next_potential = self.abstract_mdp.v_star.get(next_abstract_state, 0.0)
            shaping_reward = self.shaping_scale * (
                self.abstract_mdp.gamma * next_potential - current_potential
            )

        total_reward = task_reward + shaping_reward
        self.episode_task_reward += task_reward
        self.episode_total_reward += total_reward

        # Store one coherent record only when an episode has actually ended.
        if terminated or truncated:
            if task_success:
                end_reason = "success"
            elif environment_terminated:
                end_reason = "environment_terminated"
            else:
                end_reason = "truncated"
            self.metrics.record_episode(
                self.episode_task_reward,
                self.episode_total_reward,
                self.episode_length,
                end_reason,
                self.use_shaping,
            )

        self.current_dfa_state = next_dfa_state
        self.previous_observation = observation
        augmented_observation = self._augment_observation(
            observation,
            next_dfa_state,
        )
        return augmented_observation, total_reward, terminated, truncated, info


### `dsac_policy.py`


In [ ]:
%%writefile dsac_policy.py
"""Construction of a discrete SAC policy using Tianshou's standard networks."""

import torch
from tianshou.policy import DiscreteSACPolicy
from tianshou.utils.net.common import Net
from tianshou.utils.net.discrete import Actor, Critic


# =============================================================================
# Standard Tianshou discrete SAC networks
# =============================================================================

def build_discrete_sac_policy(
    state_shape,
    action_shape,
    device,
    learning_rate,
    hidden_sizes,
    gamma,
    alpha,
    tau,
):
    """Build DSAC with Tianshou's default MLP, actor and critic classes."""
    # The actor backbone extracts state features. Tianshou's Actor adds the
    # action head and normalizes its output into a categorical distribution.
    actor_backbone = Net(
        state_shape=state_shape,
        hidden_sizes=hidden_sizes,
        device=device,
    )
    actor = Actor(
        preprocess_net=actor_backbone,
        action_shape=action_shape,
        device=device,
        # DiscreteSACPolicy passes this output to Categorical(logits=...).
        softmax_output=False,
    ).to(device)

    # Discrete SAC uses two independent state-action value estimators to reduce
    # positive bias in the bootstrapped target.
    critic1_backbone = Net(
        state_shape=state_shape,
        hidden_sizes=hidden_sizes,
        device=device,
    )
    critic1 = Critic(
        preprocess_net=critic1_backbone,
        last_size=action_shape,
        device=device,
    ).to(device)

    critic2_backbone = Net(
        state_shape=state_shape,
        hidden_sizes=hidden_sizes,
        device=device,
    )
    critic2 = Critic(
        preprocess_net=critic2_backbone,
        last_size=action_shape,
        device=device,
    ).to(device)

    # Each network owns a separate optimizer, as required by DiscreteSACPolicy.
    policy = DiscreteSACPolicy(
        actor=actor,
        actor_optim=torch.optim.Adam(actor.parameters(), lr=learning_rate),
        critic1=critic1,
        critic1_optim=torch.optim.Adam(critic1.parameters(), lr=learning_rate),
        critic2=critic2,
        critic2_optim=torch.optim.Adam(critic2.parameters(), lr=learning_rate),
        tau=tau,
        gamma=gamma,
        alpha=alpha,
        estimation_step=1,
    )
    return policy.to(device)


### `dsac_plotting.py`


In [ ]:
%%writefile dsac_plotting.py
"""Diagnostic plots for discrete SAC experiments."""

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# =============================================================================
# Plotting helpers
# =============================================================================

def _moving_average(values, window_size):
    """Smooth an episode sequence while retaining boundary observations."""
    return pd.Series(values).rolling(
        window=window_size,
        min_periods=1,
        center=True,
    ).mean()


def plot_reward_breakdown(
    task_rewards,
    total_rewards,
    window_size=100,
    filename="img/shaping_reward_breakdown.png",
):
    """Plot synthetic task rewards against the total learning rewards."""
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    task_average = _moving_average(task_rewards, window_size)
    total_average = _moving_average(total_rewards, window_size)
    episodes = np.arange(len(task_rewards))

    figure, axis = plt.subplots(figsize=(12, 7))
    axis.plot(
        episodes,
        task_average,
        color="green",
        linewidth=2,
        label="Goal-MDP task reward",
    )
    axis.plot(
        episodes,
        total_average,
        color="purple",
        linewidth=2.5,
        label="Training reward (task + shaping)",
    )
    axis.set_title(
        f"DSAC reward analysis (moving-average window: {window_size})",
        fontsize=15,
        fontweight="bold",
    )
    axis.set_xlabel("Episode")
    axis.set_ylabel("Episode reward")
    axis.grid(True, linestyle="--", alpha=0.5)
    axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)
    figure.tight_layout()
    figure.savefig(filename, dpi=200, bbox_inches="tight")
    plt.close(figure)
    print(f"Reward plot saved to {filename}")


def plot_comparison(
    baseline_rewards,
    shaping_rewards,
    window_size=100,
    filename="img/baseline_vs_shaping.png",
):
    """Compare baseline and reward-shaped DSAC learning curves."""
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    episodes = np.arange(min(len(baseline_rewards), len(shaping_rewards)))
    figure, axis = plt.subplots(figsize=(12, 7))
    axis.plot(
        episodes,
        _moving_average(baseline_rewards[: len(episodes)], window_size),
        color="black",
        linewidth=2,
        label="DSAC without shaping",
    )
    axis.plot(
        episodes,
        _moving_average(shaping_rewards[: len(episodes)], window_size),
        color="blue",
        linewidth=2.5,
        label="DSAC with shaping",
    )
    axis.set_title("DSAC performance comparison")
    axis.set_xlabel("Episode")
    axis.set_ylabel("Episode reward")
    axis.grid(True, linestyle="--", alpha=0.5)
    axis.legend()
    figure.tight_layout()
    figure.savefig(filename, dpi=200, bbox_inches="tight")
    plt.close(figure)
    print(f"Comparison plot saved to {filename}")


### `dsac_trainer.py`


In [ ]:
%%writefile dsac_trainer.py
"""Train discrete SAC on LunarLander tasks specified by an LTLf formula."""

import argparse
import json
import os

import gymnasium as gym
import numpy as np
import torch
from tianshou.data import Collector, VectorReplayBuffer
from tianshou.env import DummyVectorEnv

from abstract_mdps import LTLfAutomaton, LTLfWaypointMDP
from automaton_validator import validate_automaton
from dsac_environment import LTLfShapingWrapper
from dsac_metrics import TrainingMetrics
from dsac_plotting import plot_reward_breakdown
from dsac_policy import build_discrete_sac_policy
from utils import save_sequential_heatmaps


# =============================================================================
# Experiment configuration
# =============================================================================

def load_experiment_config(filename):
    """Load and normalize the temporal-task configuration."""
    with open(filename, "r", encoding="utf-8") as config_file:
        raw_config = json.load(config_file)
    return {
        "formula": raw_config.get("formula", "F(goal)"),
        "waypoints": {
            name: tuple(coordinates)
            for name, coordinates in raw_config.get(
                "waypoints_dict",
                {"goal": [5, 0]},
            ).items()
        },
        "grid_width": int(raw_config.get("grid_w", 12)),
        "grid_height": int(raw_config.get("grid_h", 12)),
        "gamma": float(raw_config.get("gamma", 0.99)),
        "goal_reward": float(raw_config.get("goal_reward", 10000)),
    }


# =============================================================================
# Abstract model construction
# =============================================================================

def build_abstract_mdp(config, image_directory):
    """Build, validate and solve the abstract LTLf-guided MDP."""
    # Compile the finite-trace temporal formula into a deterministic automaton.
    automaton = LTLfAutomaton(config["formula"])

    # Reject malformed automata and waypoint definitions before allocating the
    # neural networks or starting an expensive training run.
    validation_report = validate_automaton(
        automaton,
        config["waypoints"],
        width=config["grid_width"],
        height=config["grid_height"],
    )
    print(
        "=== LTLf discrete SAC experiment ===\n"
        f"Formula: {config['formula']}\n"
        f"Waypoints: {config['waypoints']}\n"
        f"DFA states: {automaton.states}\n"
        f"Pre-trace state: {automaton.get_initial_q()}\n"
        f"Accepting states: {sorted(automaton.accepting_states)}\n"
        f"{validation_report.format()}"
    )
    automaton.render_graph(directory=image_directory)

    # Value iteration produces the potential function V*, later used by the
    # environment wrapper to compute potential-based shaping rewards.
    abstract_mdp = LTLfWaypointMDP(
        waypoints_dict=config["waypoints"],
        ltlf_automaton=automaton,
        width=config["grid_width"],
        height=config["grid_height"],
        gamma=config["gamma"],
        goal_reward=config["goal_reward"],
    )
    abstract_mdp.value_iteration()
    save_sequential_heatmaps(
        abstract_mdp,
        filename_prefix=f"{image_directory}/heatmap_V_star",
    )
    return abstract_mdp


# =============================================================================
# Environment construction
# =============================================================================

def make_environment_factory(abstract_mdp, metrics, args, goal_reward):
    """Return the environment factory required by Tianshou."""

    def make_environment():
        # A fresh base environment is required whenever the vectorized
        # environment invokes this factory.
        environment = gym.make("LunarLander-v3", continuous=False)
        return LTLfShapingWrapper(
            environment,
            abstract_mdp,
            metrics,
            use_shaping=args.use_shaping,
            shaping_scale=args.shaping_scale,
            goal_reward=goal_reward,
        )

    return make_environment


# =============================================================================
# Tianshou compatibility helpers
# =============================================================================

def _collected_steps(collection_result):
    """Read the step count from supported Tianshou collector result formats."""
    # Tianshou 0.x returns a dictionary, while later collector releases expose
    # the same information through a typed statistics object.
    if isinstance(collection_result, dict):
        return int(collection_result["n/st"])
    if hasattr(collection_result, "n_collected_steps"):
        return int(collection_result.n_collected_steps)
    raise TypeError(f"Unsupported collector result: {type(collection_result).__name__}")


# =============================================================================
# Discrete SAC training
# =============================================================================

def train_dsac(environment_factory, metrics, args, gamma):
    """Run the exact requested number of DSAC training episodes."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(
        f"\nInitializing discrete SAC on {device} "
        f"(reward shaping: {args.use_shaping})."
    )
    training_environments = DummyVectorEnv([environment_factory])

    # Infer dimensions from the wrapped environment. The observation already
    # contains the one-hot DFA state appended to the LunarLander state.
    state_shape = training_environments.observation_space[0].shape
    action_shape = training_environments.action_space[0].n
    policy = build_discrete_sac_policy(
        state_shape=state_shape,
        action_shape=action_shape,
        device=device,
        learning_rate=args.learning_rate,
        hidden_sizes=args.hidden_sizes,
        gamma=gamma,
        alpha=args.alpha,
        tau=args.tau,
    )
    replay_buffer = VectorReplayBuffer(args.buffer_size, len(training_environments))
    collector = Collector(
        policy,
        training_environments,
        replay_buffer,
        exploration_noise=True,
    )

    try:
        # Warm-up transitions populate the replay buffer only. Disabling the
        # recorder prevents partial random episodes from polluting metrics.
        print(f"Collecting {args.warmup_steps} random warm-up steps.")
        metrics.enabled = False
        collector.collect(n_step=args.warmup_steps, random=True)
        metrics.clear()
        collector.reset_env()
        metrics.enabled = True

        print(f"Training for exactly {args.episodes} episodes.")
        for _ in range(args.episodes):
            # Collect one complete episode so that experiment-level metrics
            # always contain exactly the requested number of episodes.
            collection_result = collector.collect(n_episode=1)
            gradient_steps = _collected_steps(collection_result)

            # Preserve the original update-to-data ratio: one gradient update
            # for every newly collected environment transition.
            for _ in range(gradient_steps):
                policy.update(sample_size=args.batch_size, buffer=replay_buffer)
    finally:
        metrics.enabled = False
        training_environments.close()

    metrics.validate_episode_count(args.episodes)
    print("Training completed.")
    return metrics.as_dict()


# =============================================================================
# Result persistence
# =============================================================================

def save_metrics(metrics, output_directory, use_shaping):
    """Persist numeric metrics in a format suitable for later analysis."""
    task_rewards = np.asarray(metrics["task_rewards"], dtype=np.float64)
    total_rewards = np.asarray(metrics["total_rewards"], dtype=np.float64)
    success_flags = task_rewards > 0
    transitions = sorted(metrics["dfa_transition_counts"].items())
    prefix = "shaping" if use_shaping else "baseline"
    filename = f"{output_directory}/{prefix}_dsac_data.npz"
    np.savez_compressed(
        filename,
        task_rewards=task_rewards,
        total_rewards=total_rewards,
        success_flags=success_flags,
        success_rate=float(np.mean(success_flags)) if len(success_flags) else 0.0,
        episode_lengths=np.asarray(metrics["episode_lengths"], dtype=np.int64),
        episode_end_reasons=np.asarray(metrics["episode_end_reasons"]),
        dfa_transition_labels=np.asarray(
            [f"{source}->{destination}" for (source, destination), _ in transitions],
        ),
        dfa_transition_counts=np.asarray(
            [count for _, count in transitions],
            dtype=np.int64,
        ),
        true_rewards=task_rewards,
    )
    print(f"Training metrics saved to {filename}")
    return task_rewards, total_rewards


# =============================================================================
# Experiment orchestration
# =============================================================================

def main(args):
    """Execute the complete configuration, training and reporting pipeline."""
    os.makedirs(args.output_directory, exist_ok=True)
    os.makedirs(args.image_directory, exist_ok=True)
    config = load_experiment_config(args.config)
    abstract_mdp = build_abstract_mdp(config, args.image_directory)
    metrics = TrainingMetrics(
        expected_episodes=args.episodes,
        log_interval=args.log_interval,
    )
    environment_factory = make_environment_factory(
        abstract_mdp,
        metrics,
        args,
        config["goal_reward"],
    )
    results = train_dsac(
        environment_factory,
        metrics,
        args,
        config["gamma"],
    )
    task_rewards, total_rewards = save_metrics(
        results,
        args.output_directory,
        args.use_shaping,
    )
    success_rate = float(np.mean(task_rewards > 0)) if len(task_rewards) else 0.0
    print(f"Overall success rate: {success_rate:.2%}")
    mode = "shaping" if args.use_shaping else "baseline"
    plot_reward_breakdown(
        task_rewards,
        total_rewards,
        window_size=min(args.plot_window, max(1, len(task_rewards))),
        filename=f"{args.image_directory}/dsac_{mode}_reward_breakdown.png",
    )


# =============================================================================
# Command-line interface
# =============================================================================

def build_argument_parser():
    """Define all reproducible experiment parameters exposed by the CLI."""
    parser = argparse.ArgumentParser(description="LTLf discrete SAC training.")
    parser.add_argument("--episodes", type=int, default=500)
    parser.add_argument("--config", default="trajectory.json")
    parser.add_argument(
        "--use-shaping",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Enable or disable potential-based reward shaping.",
    )
    parser.add_argument("--shaping-scale", type=float, default=1.0)
    parser.add_argument("--learning-rate", type=float, default=1e-3)
    parser.add_argument("--hidden-sizes", type=int, nargs="+", default=[128, 128])
    parser.add_argument("--alpha", type=float, default=0.05)
    parser.add_argument("--tau", type=float, default=0.005)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--buffer-size", type=int, default=100000)
    parser.add_argument("--warmup-steps", type=int, default=2000)
    parser.add_argument("--log-interval", type=int, default=100)
    parser.add_argument("--plot-window", type=int, default=50)
    parser.add_argument("--output-directory", default="results")
    parser.add_argument("--image-directory", default="img")
    return parser


if __name__ == "__main__":
    main(build_argument_parser().parse_args())


## 4. Configure the temporal task

Edit this cell to change the LTLf formula, grid or waypoint coordinates.


In [ ]:
%%writefile trajectory.json
{
    "formula": "F(wp1 & X(F(goal)))",
    "grid_w": 12,
    "grid_h": 12,
    "goal_reward": 10,
    "waypoints_dict": {
        "wp1": [1,8],
        "goal": [8,8]
    }
}


## 5. Configure the training experiment

The native LunarLander reward is ignored. Training uses the synthetic LTLf task reward and, when enabled, potential-based reward shaping.


In [ ]:
EPISODES = 2500
USE_SHAPING = True
SHAPING_SCALE = 0.25
LEARNING_RATE = 3e-4
HIDDEN_SIZES = [128, 128]
ALPHA = 0.1
TAU = 0.005
BATCH_SIZE = 128
BUFFER_SIZE = 200000
WARMUP_STEPS = 5000
LOG_INTERVAL = 100
PLOT_WINDOW = 50

print(f"Episodes: {EPISODES}")
print(f"Reward shaping: {USE_SHAPING}")
print(f"Shaping scale: {SHAPING_SCALE}")
print(f"Hidden layers: {HIDDEN_SIZES}")


## 6. Validate the reconstructed program


In [ ]:
import subprocess

SOURCE_FILES = [
    "abstract_mdps.py",
    "automaton_validator.py",
    "utils.py",
    "dsac_metrics.py",
    "dsac_environment.py",
    "dsac_policy.py",
    "dsac_plotting.py",
    "dsac_trainer.py",
]
subprocess.run(["python", "-m", "py_compile", *SOURCE_FILES], check=True)
print("All DSAC modules compiled successfully.")


## 7. Run discrete SAC training


In [ ]:
import os
import subprocess

command = [
    "python",
    "dsac_trainer.py",
    "--episodes", str(EPISODES),
    "--config", "trajectory.json",
    "--shaping-scale", str(SHAPING_SCALE),
    "--learning-rate", str(LEARNING_RATE),
    "--hidden-sizes", *map(str, HIDDEN_SIZES),
    "--alpha", str(ALPHA),
    "--tau", str(TAU),
    "--batch-size", str(BATCH_SIZE),
    "--buffer-size", str(BUFFER_SIZE),
    "--warmup-steps", str(WARMUP_STEPS),
    "--log-interval", str(LOG_INTERVAL),
    "--plot-window", str(PLOT_WINDOW),
]
command.append("--use-shaping" if USE_SHAPING else "--no-use-shaping")

environment = os.environ.copy()
environment["MPLBACKEND"] = "Agg"
print("Running:", " ".join(command), flush=True)
process = subprocess.Popen(
    command,
    cwd=WORK_DIR,
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for output_line in process.stdout:
    print(output_line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"DSAC training failed with exit code {return_code}")


## 10. Package outputs for download


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

ARCHIVE_PATH = Path("/kaggle/working/dsac_training_outputs.zip")
with ZipFile(ARCHIVE_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for directory_name in ("results", "img"):
        directory = WORK_DIR / directory_name
        if directory.exists():
            for path in sorted(directory.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(WORK_DIR))
    archive.write(WORK_DIR / "trajectory.json", "trajectory.json")

print(f"Archive ready: {ARCHIVE_PATH}")
print(f"Archive size: {ARCHIVE_PATH.stat().st_size / (1024 ** 2):.2f} MB")
